In [ ]:
###################################################################
#   UPLOAD TO GOOGLE COLAB TO EXTRACT EMBEDDINGS FOR YOUR DATASET #
###################################################################

import torch
import torch.nn as nn
import torchvision.models as models
from torchvision.models import ResNet50_Weights
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn.functional as F

#root path
root="/content"

class FeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        # Load pre-trained ResNet-50 
        resnet = models.resnet50(weights=ResNet50_Weights.DEFAULT)
        self.feature_layer = nn.Sequential(*list(resnet.children())[:-1])
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.to(self.device)

    def forward(self, x):
        with torch.no_grad():
            features = self.feature_layer(x)
            features.view(features.size(0), -1)
            features = F.normalize(features, p=2, dim=1)
        return features

    def get_or_create_embeddings(self, dataloader, save_path=f"{root}/cifar10_embeddings.pt"):
        
        features_all = []
        labels_all = []
        
        self.eval()
        with torch.no_grad():
            for images, labels in dataloader:
                images = images.to(self.device)
                feats = self.forward(images)
                features_all.append(feats.cpu())
                labels_all.append(labels)
        
        features_cat = torch.cat(features_all)
        labels_cat = torch.cat(labels_all)
        
        # Save the dictionary locally for next time
        torch.save({'features': features_cat, 'labels': labels_cat}, save_path)
        print(f"--- Extraction complete and saved to {save_path} ---")
        
        return features_cat, labels_cat

def get_dataloader(batch_size=64):
    """
    Prepares and returns the CIFAR-10 data loader.
    Includes resizing and normalization for ResNet-50 compatibility.
    """
    # ResNet-50 expects 224x224 images and specific normalization
    transform = transforms.Compose([
        transforms.Resize((224, 224)), 
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406], # ImageNet standards
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    # Load the CIFAR-10 dataset
    # 'train=False' loads the test set (10,000 images), which is great for testing
    dataset = datasets.CIFAR10(
        root=f"{root}/cifar10", 
        train=True, 
        download=True, 
        transform=transform
    )
    
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    return loader


FE=FeatureExtractor()
load=get_dataloader(batch_size=256)
FE.get_or_create_embeddings(dataloader=load)

--- Found local embeddings at ../embeddings/cifar10_embeddings.pt. Loading... ---


(tensor([[0.0000e+00, 2.5539e-01, 0.0000e+00,  ..., 0.0000e+00, 2.0336e-01,
          5.1078e-02],
         [2.9832e-01, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
          0.0000e+00],
         [7.8000e-02, 8.0727e-02, 2.0732e-02,  ..., 6.4685e-02, 0.0000e+00,
          1.1618e+00],
         ...,
         [4.0346e-01, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 1.4277e-02,
          1.9108e-02],
         [1.8029e-02, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
          0.0000e+00],
         [8.0387e-04, 0.0000e+00, 0.0000e+00,  ..., 1.5782e-02, 0.0000e+00,
          0.0000e+00]]),
 tensor([6, 9, 9,  ..., 9, 1, 1]))